# RankLab: real KuaiRand-1K catalog scale benchmark

This notebook measures exact versus HNSW index recall and latency on the attached actual 1K item catalog. It is not a trained recommender-quality benchmark.

In [ ]:
from pathlib import Path
import subprocess

repo = Path('/kaggle/working/rank-lab')
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/kushc2004/rank-lab.git', str(repo)], check=True)
subprocess.run(['pip', 'install', '--quiet', '-e', '.[full]'], cwd=repo, check=True)

def find_1k_root() -> Path:
    required = 'log_standard_4_08_to_4_21_1k.csv'
    matches = list(Path('/kaggle/input').rglob(required))
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one attached official 1K standard log, found: {matches}')
    return matches[0].parent

raw_dir = find_1k_root()
print(f'Raw 1K data: {raw_dir}', flush=True)
subprocess.run(['python', 'scripts/audit_kuairand_1k.py', '--raw-dir', str(raw_dir)], cwd=repo, check=True)


In [ ]:
subprocess.run([
    'python', 'scripts/benchmark_kuairand_1k_scale.py',
    '--raw-dir', str(raw_dir), '--output', 'outputs/metrics/kuairand_1k_scale.json',
    '--dimension', '64', '--query-count', '100', '--candidate-k', '200',
], cwd=repo, check=True)
subprocess.run(['tar', '-czf', '/kaggle/working/ranklab_kuairand_1k_outputs.tar.gz', '-C', str(repo), 'outputs'], check=True)
print('Saved /kaggle/working/ranklab_kuairand_1k_outputs.tar.gz', flush=True)
